# Imports

In [ ]:
import json
import urllib.error
import urllib.request
from concurrent.futures import ThreadPoolExecutor
from io import BytesIO
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

from uavloc.datasets import DERIVED, data_root
from uavloc.matching.registration import apply_shift, phase_shift
from uavloc.track import REPO_ROOT

# Params

In [ ]:
DATASET = "ua-village-multidate"
N_VILLAGES = 8
N_DETAIL_VILLAGES = 4
DETAIL_ZOOMS = [18, 19, 20]
DETAIL_BLOCK_Z18_TILES = 2
SESSION_REGION = "UA"
WORKERS = 8
MAX_NEW_TILES = 3000
APPEARANCE_MAX_CORR = 0.97
MIN_SHIFT_PEAK = 15.0
MAX_SHIFT_PX = 30.0
DETAIL_CROP_FRACTION = 0.25

# Data

In [ ]:
TILES_API = "https://tile.googleapis.com"
TILE_PX = 256
ENV_FILE = REPO_ROOT / ".env"
TILE_CACHE = DERIVED / "tile-cache" / "google-satellite"
POC_DIR = REPO_ROOT / "pocs" / "google-satellite"
FIGURES = POC_DIR / "figures"
RESULTS = POC_DIR / "results.json"
FIGURES.mkdir(exist_ok=True)

In [ ]:
def load_api_key(env_file: Path) -> str:
    for line in env_file.read_text().splitlines():
        name, _, value = line.partition("=")
        if name.strip() == "GOOGLE_MAPS_API_KEY":
            return value.strip().strip('"')
    raise KeyError(f"GOOGLE_MAPS_API_KEY not in {env_file}")

In [ ]:
def create_session(api_key: str, region: str) -> str:
    body = json.dumps({"mapType": "satellite", "language": "en-US", "region": region}).encode()
    request = urllib.request.Request(f"{TILES_API}/v1/createSession?key={api_key}", data=body,
                                     headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(request, timeout=30) as response:
        return json.load(response)["session"]

In [ ]:
API_KEY = load_api_key(ENV_FILE)
SESSION = create_session(API_KEY, SESSION_REGION)

In [ ]:
def tile_path(z: int, x: int, y: int) -> Path:
    return TILE_CACHE / str(z) / str(x) / f"{y}.jpg"

In [ ]:
def download(url: str, retries: int = 3) -> bytes:
    """Response body; b"" for 404, which the Map Tiles API returns where it has no imagery."""
    for attempt in range(retries):
        try:
            with urllib.request.urlopen(url, timeout=30) as response:
                return response.read()
        except urllib.error.HTTPError as exc:
            if exc.code == 404:
                return b""
            if attempt == retries - 1:
                raise
        except (urllib.error.URLError, TimeoutError):
            if attempt == retries - 1:
                raise

In [ ]:
def fetch_tile(z: int, x: int, y: int) -> bytes | None:
    """One Google satellite tile, from the disk cache if present. `None` means no imagery."""
    path = tile_path(z, x, y)
    if not path.exists():
        data = download(f"{TILES_API}/v1/2dtiles/{z}/{x}/{y}?session={SESSION}&key={API_KEY}")
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(data)
    return path.read_bytes() or None

In [ ]:
def grid_tiles(grid: dict) -> list[tuple[int, int, int]]:
    return [(grid["z"], x, y)
            for y in range(grid["y0"], grid["y0"] + grid["ny"])
            for x in range(grid["x0"], grid["x0"] + grid["nx"])]

In [ ]:
def grid_at_zoom(grid: dict, z: int) -> dict:
    """The same ground as `grid`, in tiles of zoom `z >= grid["z"]`."""
    k = 2 ** (z - grid["z"])
    return {"z": z, "x0": grid["x0"] * k, "y0": grid["y0"] * k, "nx": grid["nx"] * k, "ny": grid["ny"] * k}

In [ ]:
def centre_block(grid: dict, side: int) -> dict:
    """A `side` x `side` tile block at the centre of `grid`, same zoom."""
    return {"z": grid["z"], "x0": grid["x0"] + (grid["nx"] - side) // 2,
            "y0": grid["y0"] + (grid["ny"] - side) // 2, "nx": side, "ny": side}

In [ ]:
def mosaic(grid: dict) -> np.ndarray:
    """Every tile of `grid` stitched into (H, W, 3) uint8. Missing tiles stay black."""
    out = np.zeros((grid["ny"] * TILE_PX, grid["nx"] * TILE_PX, 3), dtype=np.uint8)
    tiles = grid_tiles(grid)
    with ThreadPoolExecutor(WORKERS) as pool:
        for (_, x, y), data in zip(tiles, pool.map(lambda t: fetch_tile(*t), tiles)):
            if data:
                r, c = (y - grid["y0"]) * TILE_PX, (x - grid["x0"]) * TILE_PX
                out[r:r + TILE_PX, c:c + TILE_PX] = np.array(Image.open(BytesIO(data)).convert("RGB"))
    return out

In [ ]:
def load_villages(n: int) -> list[dict]:
    """`n` villages spread evenly over the dataset's sorted folders, each its meta.json plus `key`."""
    paths = sorted(data_root(DATASET).glob("*/meta.json"))
    picks = np.linspace(0, len(paths) - 1, n).round().astype(int)
    return [json.loads(paths[i].read_text()) | {"key": paths[i].parent.name} for i in picks]

In [ ]:
def load_esri(village: dict) -> dict[str, np.ndarray]:
    """Every stored Esri date of `village`, already co-registered to its reference by the build."""
    folder = data_root(DATASET) / village["key"]
    return {date: np.array(Image.open(folder / f"{date}.jpg").convert("RGB")) for date in village["dates"]}

In [ ]:
VILLAGES = load_villages(N_VILLAGES)
DETAIL_GRIDS = [grid_at_zoom(centre_block(v["grid"], DETAIL_BLOCK_Z18_TILES), z)
                for v in VILLAGES[:N_DETAIL_VILLAGES] for z in DETAIL_ZOOMS]
FULL_GRIDS = [v["grid"] for v in VILLAGES]
[v["key"] for v in VILLAGES]

In [ ]:
def new_tile_count(grids: list[dict]) -> int:
    return len({t for g in grids for t in grid_tiles(g) if not tile_path(*t).exists()})

In [ ]:
n_new = new_tile_count(FULL_GRIDS + DETAIL_GRIDS)
if n_new > MAX_NEW_TILES:
    raise RuntimeError(f"{n_new} uncached tiles exceeds MAX_NEW_TILES={MAX_NEW_TILES}")
n_new

# Model

In [ ]:
def grey(rgb: np.ndarray) -> np.ndarray:
    return rgb.astype(np.float32).mean(axis=2)

In [ ]:
def correlation(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.corrcoef(a.ravel(), b.ravel())[0, 1])

In [ ]:
def compare_to_esri(village: dict, google: np.ndarray) -> dict:
    """Shift taking Google onto the Esri reference, and its correlation with every Esri date after it."""
    esri = load_esri(village)
    dx, dy, peak = phase_shift(grey(esri[village["reference_date"]]), grey(google))
    aligned = grey(apply_shift(google, dx, dy))
    corr = {date: correlation(aligned, grey(rgb)) for date, rgb in esri.items()}
    max_corr = max(corr.values())
    return {"key": village["key"], "shift_px": [dx, dy], "shift_m": float(np.hypot(dx, dy) * village["mpp"]),
            "peak_ratio": peak, "shift_reliable": bool(peak >= MIN_SHIFT_PEAK and np.hypot(dx, dy) <= MAX_SHIFT_PX),
            "coverage": float(google.any(axis=2).mean()), "corr_vs_esri": corr, "max_corr": max_corr,
            "closest_esri_date": max(corr, key=corr.get), "distinct": bool(max_corr < APPEARANCE_MAX_CORR)}

In [ ]:
def detail_beyond_half(rgb: np.ndarray) -> float:
    """Share of variance lost by halving then restoring resolution; near 0 means the image is upsampled."""
    g = grey(rgb)
    h, w = g.shape
    small = Image.fromarray(g).resize((w // 2, h // 2), Image.BOX)
    back = np.asarray(small.resize((w, h), Image.BICUBIC), dtype=np.float32)
    return float(((g - back) ** 2).mean() / g.var())

In [ ]:
def esri_block(village: dict, block: dict) -> np.ndarray:
    """The Esri reference pixels under a z18 tile block of the village grid."""
    esri = np.asarray(Image.open(data_root(DATASET) / village["key"] / f"{village['reference_date']}.jpg"))
    r, c = (block["y0"] - village["grid"]["y0"]) * TILE_PX, (block["x0"] - village["grid"]["x0"]) * TILE_PX
    return esri[r:r + block["ny"] * TILE_PX, c:c + block["nx"] * TILE_PX]

In [ ]:
def detail_by_zoom(village: dict) -> dict:
    """`detail_beyond_half` for Google at each zoom and for Esri z18, on the same central ground."""
    block = centre_block(village["grid"], DETAIL_BLOCK_Z18_TILES)
    google = {f"google_z{z}": detail_beyond_half(mosaic(grid_at_zoom(block, z))) for z in DETAIL_ZOOMS}
    return {"key": village["key"], **google, "esri_z18": detail_beyond_half(esri_block(village, block))}

# Run

In [ ]:
comparisons = [compare_to_esri(v, mosaic(v["grid"])) for v in VILLAGES]

In [ ]:
details = [detail_by_zoom(v) for v in VILLAGES[:N_DETAIL_VILLAGES]]

In [ ]:
alignment = pd.DataFrame(comparisons).drop(columns=["corr_vs_esri", "shift_px"])
alignment

In [ ]:
alignment[["shift_m", "peak_ratio", "max_corr", "coverage"]].describe()

In [ ]:
detail = pd.DataFrame(details).set_index("key")
detail

In [ ]:
n_chars = RESULTS.write_text(json.dumps({"villages": comparisons, "detail": details,
                               "cached_tiles": sum(1 for _ in TILE_CACHE.rglob("*.jpg"))}, indent=2))

# Plots

In [ ]:
def plot_side_by_side(village: dict, comparison: dict, axes) -> None:
    """Esri reference, then Google z18 and z19 shifted onto it, over the central tile block."""
    block = centre_block(village["grid"], DETAIL_BLOCK_Z18_TILES)
    dx, dy = comparison["shift_px"]
    images = [esri_block(village, block), apply_shift(mosaic(block), dx, dy),
              apply_shift(mosaic(grid_at_zoom(block, 19)), 2 * dx, 2 * dy)]
    titles = [f"Esri {village['reference_date']} z18",
              f"Google z18 (shift {comparison['shift_m']:.1f} m, r={comparison['max_corr']:.2f})", "Google z19"]
    for ax, image, title in zip(axes, images, titles):
        ax.imshow(image)
        ax.set_title(title, fontsize=9)
        ax.set_xticks([]), ax.set_yticks([])
    axes[0].set_ylabel(village["key"], fontsize=9)

In [ ]:
fig, axes = plt.subplots(N_DETAIL_VILLAGES, 3, figsize=(13, 4.4 * N_DETAIL_VILLAGES))
for row, village, comparison in zip(axes, VILLAGES, comparisons):
    plot_side_by_side(village, comparison, row)
fig.tight_layout()
fig.savefig(FIGURES / "esri-vs-google.png", dpi=110)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
detail.rename(index=lambda key: key.split("-")[0]).plot.bar(ax=ax, rot=0)
ax.set_ylabel("variance lost by 2x down/up")
ax.set_title("Detail in the top octave of each image (near 0 = upsampled)", fontsize=10)
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left")
fig.tight_layout()
fig.savefig(FIGURES / "detail-by-zoom.png", dpi=110)

In [ ]:
def centre_square(rgb: np.ndarray, fraction: float) -> np.ndarray:
    h, w = rgb.shape[:2]
    side = int(min(h, w) * fraction)
    r, c = (h - side) // 2, (w - side) // 2
    return rgb[r:r + side, c:c + side]

In [ ]:
def plot_native_pixels(village: dict, axes) -> None:
    """The same ~50 m of ground at each source's native pixels, drawn without smoothing."""
    block = centre_block(village["grid"], DETAIL_BLOCK_Z18_TILES)
    images = {"Esri z18": esri_block(village, block)}
    images |= {f"Google z{z}": mosaic(grid_at_zoom(block, z)) for z in DETAIL_ZOOMS}
    for ax, (title, image) in zip(axes, images.items()):
        crop = centre_square(image, DETAIL_CROP_FRACTION)
        ax.imshow(crop, interpolation="nearest")
        ax.set_title(f"{title}: {crop.shape[0]} px", fontsize=9)
        ax.set_xticks([]), ax.set_yticks([])
    axes[0].set_ylabel(village["key"].split("-")[0], fontsize=9)

In [ ]:
fig, axes = plt.subplots(N_DETAIL_VILLAGES, 1 + len(DETAIL_ZOOMS), figsize=(14, 3.6 * N_DETAIL_VILLAGES))
for row, village in zip(axes, VILLAGES):
    plot_native_pixels(village, row)
fig.tight_layout()
fig.savefig(FIGURES / "native-pixels.png", dpi=110)